#### - IMPORTS

In [1]:

import os
import json
import random
import time
import platform
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F

try:
    from sklearn.metrics import confusion_matrix, classification_report
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("[INFO] scikit-learn not available — confusion matrix skipped.")

#### - CONFIG / HYPERPARAMETERS

In [2]:

CONFIG = {
    "seed": 42,

    # ── Dataset ───────────────────────────────────────────────────────────
    "dataset": "MNIST",
    "data_dir": "./data",
    "num_workers": 2,

    # ── Training ──────────────────────────────────────────────────────────
    "batch_size": 64,
    "learning_rate": 1e-3,
    "num_epochs": 10,
    "weight_decay": 1e-4,

    # ── Architecture ──────────────────────────────────────────────────────
    # Conv layers: (out_channels, kernel_size, stride, padding)
    "conv1": (32, 3, 1, 1),
    "conv2": (64, 3, 1, 1),
    "fc1_units": 256,
    "dropout_rate": 0.5,

    # ── Model variant ─────────────────────────────────────────────────────
    "model_variant": "dual",

    # ── I/O ───────────────────────────────────────────────────────────────
    "model_save_path": "./cnn_bns_hw_baseline.pth",
    "metadata_save_path": "./cnn_bns_hw_baseline_metadata.json",

    # ── Device ────────────────────────────────────────────────────────────
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print(f"[CONFIG] Device  : {CONFIG['device']}")
print(f"[CONFIG] Dataset : {CONFIG['dataset']}")
print(f"[CONFIG] Variant : {CONFIG['model_variant']}")

[CONFIG] Device  : cpu
[CONFIG] Dataset : MNIST
[CONFIG] Variant : dual


#### - REPRODUCIBILITY

In [3]:
def set_seed(seed: int) -> None:
    """
    Sets all random seeds for full reproducibility.
    Required so that weight initialization is identical across BNS/RNS runs.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[SEED] All seeds set to {seed}")


set_seed(CONFIG["seed"])


[SEED] All seeds set to 42


#### - EXPERIMENT METADATA

In [4]:
@dataclass
class ExperimentMetadata:
    run_id:           str   = ""
    model_variant:    str   = "dual"
    config:           dict  = field(default_factory=dict)
    architecture:     dict  = field(default_factory=dict)
    total_params:     int   = 0
    training_history: list  = field(default_factory=list)
    final_test_acc:   float = 0.0
    pytorch_version:  str   = torch.__version__
    python_version:   str   = platform.python_version()
    hardware:         str   = ""
    timestamp:        str   = ""
    notes:            str   = (
        "Hardware-friendly baseline: BatchNorm removed, bias=True on Conv2d. "
        "Raw conv outputs (pre-ReLU, pre-Pool) captured for RNS comparison."
    )

    def __post_init__(self):
        if not self.run_id:
            self.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        if not self.timestamp:
            self.timestamp = datetime.now().isoformat()
        if not self.hardware:
            if torch.cuda.is_available():
                self.hardware = torch.cuda.get_device_name(0)
            else:
                self.hardware = platform.processor() or "CPU"

    def save(self, path: str) -> None:
        """Serializes metadata to a human-readable JSON file."""
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2)
        print(f"[META] Experiment metadata saved to: {path}")

    @classmethod
    def load(cls, path: str) -> "ExperimentMetadata":
        """Loads metadata from a JSON file."""
        with open(path, "r") as f:
            data = json.load(f)
        return cls(**data)

#### - DATASET LOADING

In [5]:
def get_dataloaders(config: dict):
    name = config["dataset"]

    if name == "MNIST":
        mean, std   = (0.1307,), (0.3081,)
        input_shape = (1, 28, 28)
        num_classes = 10
        tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_ds = datasets.MNIST(config["data_dir"], train=True,  download=True, transform=tf)
        test_ds  = datasets.MNIST(config["data_dir"], train=False, download=True, transform=tf)

    elif name == "CIFAR10":
        mean, std   = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
        input_shape = (3, 32, 32)
        num_classes = 10
        tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_ds = datasets.CIFAR10(config["data_dir"], train=True,  download=True, transform=tf)
        test_ds  = datasets.CIFAR10(config["data_dir"], train=False, download=True, transform=tf)

    else:
        raise ValueError(f"Unknown dataset: {name}")

    pin = (config["device"] == "cuda")

    train_loader = DataLoader(
        train_ds, batch_size=config["batch_size"], shuffle=True,
        num_workers=config["num_workers"], pin_memory=pin,
        worker_init_fn=lambda wid: np.random.seed(config["seed"] + wid),
    )
    test_loader = DataLoader(
        test_ds, batch_size=config["batch_size"], shuffle=False,
        num_workers=config["num_workers"], pin_memory=pin,
    )

    print(f"[DATA] Train: {len(train_ds)}  |  Test: {len(test_ds)}")
    print(f"[DATA] Batch size: {config['batch_size']}  |  Input: {input_shape}  |  Classes: {num_classes}")

    return train_loader, test_loader, num_classes, input_shape

#### - HARDWARE-FRIENDLY CONV BLOCK

In [6]:
class HWConvBlock(nn.Module):

  def __init__(
      self,
      in_channels:  int,
      out_channels: int,
      kernel_size:  int = 3,
      stride:       int = 1,
      padding:      int = 1,
      pool_size:    int = 2,
  ):
      super().__init__()
      self.conv = nn.Conv2d(
          in_channels, out_channels,
          kernel_size=kernel_size,
          stride=stride,
          padding=padding,
          bias=True,
      )

      self.relu = nn.ReLU(inplace=False)   # inplace=False preserves pre-relu tensor
      self.pool = nn.MaxPool2d(kernel_size=pool_size, stride=pool_size)
      nn.init.kaiming_normal_(self.conv.weight, mode='fan_out', nonlinearity='relu')
      nn.init.zeros_(self.conv.bias)

  def forward(
      self,
      x: torch.Tensor,
      return_raw: bool = False,
  ):

      # Step 1 — Convolution (pure MAC + bias)
      after_conv = self.conv(x)

      # Step 2 — ReLU
      after_relu = self.relu(after_conv)

      # Step 3 — MaxPool
      after_pool = self.pool(after_relu)

      if return_raw:
          raw = {
              "pre_relu"  : after_conv.detach().clone(),  # ← RNS must match this
              "pre_pool"  : after_relu.detach().clone(),
              "post_pool" : after_pool.detach().clone(),
          }
          return after_pool, raw

      return after_pool


#### - DUAL-CONV MODEL  (default — matches paper architecture)

In [7]:
class CNN_BNS_HW(nn.Module):
    def __init__(
        self,
        in_channels:  int,
        num_classes:  int,
        conv1_cfg:    tuple = (32, 3, 1, 1),
        conv2_cfg:    tuple = (64, 3, 1, 1),
        fc1_units:    int   = 256,
        dropout_rate: float = 0.5,
        input_hw:     tuple = (28, 28),
    ):
        super().__init__()

        c1_out, c1_k, c1_s, c1_p = conv1_cfg
        c2_out, c2_k, c2_s, c2_p = conv2_cfg

        self.conv_block1 = HWConvBlock(in_channels, c1_out, c1_k, c1_s, c1_p)
        self.conv_block2 = HWConvBlock(c1_out,      c2_out, c2_k, c2_s, c2_p)

        h_out    = input_hw[0] // 4
        w_out    = input_hw[1] // 4
        flat_dim = c2_out * h_out * w_out

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, fc1_units),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(fc1_units, num_classes),
        )

        # Init FC layers
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        self._arch_summary = {
            "type":         "CNN_BNS_HW (dual-conv)",
            "batch_norm":   False,
            "conv_bias":    True,
            "conv_block1":  f"{in_channels}→{c1_out} ch, k={c1_k}, pad={c1_p}",
            "conv_block2":  f"{c1_out}→{c2_out} ch, k={c2_k}, pad={c2_p}",
            "flatten_dim":  flat_dim,
            "fc1":          f"{flat_dim}→{fc1_units}",
            "fc2_out":      f"{fc1_units}→{num_classes}",
            "total_params": total_params,
        }

        print(f"\n[MODEL] CNN_BNS_HW — Dual ConvBlock (no BatchNorm, bias=True)")
        for k, v in self._arch_summary.items():
            print(f"  {k:<16}: {v}")
        print()

    def forward(
        self,
        x:               torch.Tensor,
        return_features: bool = False,
    ):
        fmaps = {}

        if return_features:
            fmaps["input"] = x.detach().clone()

        # ── Conv Block 1 ──────────────────────────────────────────────────
        if return_features:
            x, raw1 = self.conv_block1(x, return_raw=True)
            fmaps["conv1.pre_relu"]  = raw1["pre_relu"]
            fmaps["conv1.pre_pool"]  = raw1["pre_pool"]
            fmaps["conv1.post_pool"] = raw1["post_pool"]
        else:
            x = self.conv_block1(x)

        # ── Conv Block 2 ──────────────────────────────────────────────────
        if return_features:
            x, raw2 = self.conv_block2(x, return_raw=True)
            fmaps["conv2.pre_relu"]  = raw2["pre_relu"]
            fmaps["conv2.pre_pool"]  = raw2["pre_pool"]
            fmaps["conv2.post_pool"] = raw2["post_pool"]
        else:
            x = self.conv_block2(x)

        # ── Flatten capture ───────────────────────────────────────────────
        if return_features:
            fmaps["flatten"] = x.detach().clone().view(x.size(0), -1)

        # ── Classifier ────────────────────────────────────────────────────
        logits = self.classifier(x)

        if return_features:
            return logits, fmaps
        return logits


#### - SINGLE-CONV MODEL  (paper-minimal variant for RNS)

In [8]:
class SingleConvCNN_BNS(nn.Module):

    def __init__(
        self,
        in_channels:  int   = 1,
        num_classes:  int   = 10,
        conv_cfg:     tuple = (32, 3, 1, 1),   # (out_ch, k, stride, pad)
        fc1_units:    int   = 128,
        dropout_rate: float = 0.5,
        input_hw:     tuple = (28, 28),
    ):
        super().__init__()

        c_out, c_k, c_s, c_p = conv_cfg

        self.conv_block = HWConvBlock(in_channels, c_out, c_k, c_s, c_p)

        # After one MaxPool2d(2,2): H→H/2, W→W/2
        h_out    = input_hw[0] // 2
        w_out    = input_hw[1] // 2
        flat_dim = c_out * h_out * w_out

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, fc1_units),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            nn.Linear(fc1_units, num_classes),
        )

        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        self._arch_summary = {
            "type":        "SingleConvCNN_BNS",
            "batch_norm":  False,
            "conv_bias":   True,
            "conv_block":  f"{in_channels}→{c_out} ch, k={c_k}, pad={c_p}",
            "flatten_dim": flat_dim,
            "fc1":         f"{flat_dim}→{fc1_units}",
            "fc2_out":     f"{fc1_units}→{num_classes}",
            "total_params": total_params,
        }

        print(f"\n[MODEL] SingleConvCNN_BNS (paper-minimal, no BatchNorm, bias=True)")
        for k, v in self._arch_summary.items():
            print(f"  {k:<16}: {v}")
        print()

    def forward(
        self,
        x:               torch.Tensor,
        return_features: bool = False,
    ):
        fmaps = {}

        if return_features:
            fmaps["input"] = x.detach().clone()
            x, raw = self.conv_block(x, return_raw=True)
            fmaps["conv.pre_relu"]  = raw["pre_relu"]
            fmaps["conv.pre_pool"]  = raw["pre_pool"]
            fmaps["conv.post_pool"] = raw["post_pool"]
            fmaps["flatten"] = x.detach().clone().view(x.size(0), -1)
        else:
            x = self.conv_block(x)

        logits = self.classifier(x)

        if return_features:
            return logits, fmaps
        return logits


#### - MODEL FACTORY

In [9]:
def build_model(config: dict, num_classes: int, input_shape: tuple) -> nn.Module:
    in_ch    = input_shape[0]
    input_hw = (input_shape[1], input_shape[2])
    variant  = config["model_variant"]

    if variant == "dual":
        return CNN_BNS_HW(
            in_channels  = in_ch,
            num_classes  = num_classes,
            conv1_cfg    = config["conv1"],
            conv2_cfg    = config["conv2"],
            fc1_units    = config["fc1_units"],
            dropout_rate = config["dropout_rate"],
            input_hw     = input_hw,
        )
    elif variant == "single":
        return SingleConvCNN_BNS(
            in_channels  = in_ch,
            num_classes  = num_classes,
            conv_cfg     = config["conv1"],
            fc1_units    = 128,
            dropout_rate = config["dropout_rate"],
            input_hw     = input_hw,
        )
    else:
        raise ValueError(f"Unknown model_variant: '{variant}'. Use 'dual' or 'single'.")


#### - TRAINING LOOP

In [10]:
def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device:    torch.device,
    epoch:     int,
    num_epochs: int,
) -> tuple:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    log_every = max(1, len(loader) // 10)   # Print 10× per epoch

    for i, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += images.size(0)

        if (i + 1) % log_every == 0:
            print(f"  Epoch [{epoch}/{num_epochs}] "
                  f"Batch [{i+1}/{len(loader)}] "
                  f"Loss: {loss.item():.4f} | "
                  f"Running Acc: {100.*correct/total:.2f}%")

    return total_loss / total, 100.0 * correct / total


def train(
    model:        nn.Module,
    train_loader: DataLoader,
    test_loader:  DataLoader,
    config:       dict,
    metadata:     ExperimentMetadata,
) -> dict:
    device    = torch.device(config["device"])
    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    history       = {"train_loss": [], "train_acc": [], "test_acc": []}
    best_test_acc = 0.0

    print("=" * 65)
    print(f"  Training — {config['num_epochs']} epochs | variant: {config['model_variant']}")
    print("=" * 65)

    for epoch in range(1, config["num_epochs"] + 1):
        t0 = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device,
            epoch, config["num_epochs"]
        )
        test_acc, _ = evaluate(model, test_loader, device)
        scheduler.step(test_acc)

        elapsed = time.time() - t0
        history["train_loss"].append(round(train_loss, 6))
        history["train_acc"].append(round(train_acc, 4))
        history["test_acc"].append(round(test_acc, 4))

        # ── Append to metadata history ─────────────────────────────────
        metadata.training_history.append({
            "epoch":      epoch,
            "train_loss": round(train_loss, 6),
            "train_acc":  round(train_acc, 4),
            "test_acc":   round(test_acc, 4),
            "lr":         round(optimizer.param_groups[0]["lr"], 8),
        })

        print(f"\n[Epoch {epoch:>2}/{config['num_epochs']}] "
              f"Loss: {train_loss:.4f} | "
              f"Train: {train_acc:.2f}% | "
              f"Test: {test_acc:.2f}% | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
              f"Time: {elapsed:.1f}s")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            save_model(model, config["model_save_path"])
            print(f"  ✓ Best model saved ({best_test_acc:.2f}%)")

        print("-" * 65)

    metadata.final_test_acc = best_test_acc
    print(f"\n[TRAINING COMPLETE] Best Test Acc: {best_test_acc:.2f}%")
    return history

#### - EVALUATION

In [11]:
def evaluate(
    model:   nn.Module,
    loader:  DataLoader,
    device:  torch.device,
    verbose: bool = False,
) -> tuple:
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            preds  = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = 100.0 * correct / total

    if verbose:
        print(f"\n[EVAL] Accuracy: {acc:.2f}% ({correct}/{total})")
        if SKLEARN_AVAILABLE:
            print("\n[CLASSIFICATION REPORT]")
            print(classification_report(all_labels, all_preds, digits=4))
            print("\n[CONFUSION MATRIX]")
            print(confusion_matrix(all_labels, all_preds))

    return acc, all_preds


def show_sample_predictions(
    model:       nn.Module,
    loader:      DataLoader,
    device:      torch.device,
    num_samples: int = 10,
) -> None:

    model.eval()
    images, labels = next(iter(loader))
    images = images[:num_samples].to(device)
    labels = labels[:num_samples]

    with torch.no_grad():
        logits = model(images)
        probs  = F.softmax(logits, dim=1)
        preds  = logits.argmax(1).cpu()
        confs  = probs.max(1).values.cpu()

    print("\n[SAMPLE PREDICTIONS]")
    print(f"{'Sample':<8} {'GT':>4} {'Pred':>6} {'Conf%':>8} {'OK?':>5}")
    print("-" * 38)
    for i in range(num_samples):
        mark = "✓" if preds[i] == labels[i] else "✗"
        print(f"{i:<8} {labels[i].item():>4} {preds[i].item():>6} "
              f"{confs[i].item()*100:>7.2f}% {mark:>5}")

#### - HARDWARE INFERENCE FUNCTION  (BNS reference)

In [12]:
def inference_bns(
    model:           nn.Module,
    input_tensor:    torch.Tensor,
    device:          torch.device,
    return_features: bool = False,
) -> dict:
    model.eval()

    if input_tensor.dim() == 3:
        input_tensor = input_tensor.unsqueeze(0)

    input_tensor = input_tensor.to(device)

    with torch.no_grad():
        if return_features:
            logits, feature_maps = model(input_tensor, return_features=True)
        else:
            logits       = model(input_tensor)
            feature_maps = None

    probs  = F.softmax(logits, dim=1)
    result = {
        "logits"          : logits.cpu(),
        "probabilities"   : probs.cpu(),
        "predicted_class" : logits.argmax(1).cpu(),
        "confidence"      : probs.max(1).values.cpu(),
    }
    if return_features:
        result["feature_maps"] = {k: v.cpu() for k, v in feature_maps.items()}

    return result

#### - MODEL SAVE / LOAD

In [13]:
def save_model(model: nn.Module, path: str) -> None:
    """Saves model state_dict to .pth file."""
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
    torch.save(model.state_dict(), path)
    print(f"[SAVE] Weights → {path}")


def load_model(model: nn.Module, path: str, device: torch.device) -> nn.Module:
    """Loads state_dict into model and sets eval mode."""
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    print(f"[LOAD] Weights ← {path}")
    return model

#### - MAIN ENTRY POINT

In [14]:
def main():
    device = torch.device(CONFIG["device"])

    # ── 1. Metadata object (collects info throughout run) ─────────────────
    metadata = ExperimentMetadata(
        model_variant = CONFIG["model_variant"],
        config        = {k: str(v) for k, v in CONFIG.items()},  # JSON-safe
    )

    # ── 2. Dataset ────────────────────────────────────────────────────────
    train_loader, test_loader, num_classes, input_shape = get_dataloaders(CONFIG)

    # ── 3. Model ──────────────────────────────────────────────────────────
    model = build_model(CONFIG, num_classes, input_shape)
    metadata.architecture  = model._arch_summary
    metadata.total_params  = model._arch_summary["total_params"]

    # ── 4. Train ──────────────────────────────────────────────────────────
    train(model, train_loader, test_loader, CONFIG, metadata)

    # ── 5. Load best checkpoint ───────────────────────────────────────────
    model = load_model(model, CONFIG["model_save_path"], device)

    # ── 6. Final evaluation ───────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  FINAL EVALUATION")
    print("=" * 65)
    final_acc, _ = evaluate(model, test_loader, device, verbose=True)
    show_sample_predictions(model, test_loader, device, num_samples=10)

    # ── 7. Feature map extraction + RNS validation demo ───────────────────
    print("\n[FEATURE MAP DEMO] Pre-ReLU (RNS target) + full capture")
    sample_images, sample_labels = next(iter(test_loader))
    sample_input = sample_images[:1].to(device)

    result = inference_bns(model, sample_input, device, return_features=True)
    fmaps  = result["feature_maps"]

    print(f"\n  {'Key':<22} {'Shape':<28} {'Min':>10} {'Max':>10} {'Mean':>10}")
    print(f"  {'-'*82}")
    for key, tensor in fmaps.items():
        t = tensor.float()
        print(f"  {key:<22} {str(tuple(t.shape)):<28} "
              f"{t.min().item():>10.4f} {t.max().item():>10.4f} {t.mean().item():>10.4f}")

    print(f"\n  Predicted class : {result['predicted_class'].item()}")
    print(f"  Ground truth    : {sample_labels[0].item()}")
    print(f"  Confidence      : {result['confidence'].item()*100:.2f}%")

    # ── 8. Training history table ─────────────────────────────────────────
    print("\n[TRAINING HISTORY]")
    print(f"{'Ep':>4} {'Train Loss':>12} {'Train Acc%':>12} {'Test Acc%':>11} {'LR':>12}")
    print("-" * 55)
    for row in metadata.training_history:
        print(f"{row['epoch']:>4} {row['train_loss']:>12.4f} "
              f"{row['train_acc']:>12.2f} {row['test_acc']:>11.2f} "
              f"{row['lr']:>12.2e}")

    # ── 9. Save metadata ──────────────────────────────────────────────────
    metadata.final_test_acc = final_acc
    metadata.save(CONFIG["metadata_save_path"])

    # ── 10. Summary ───────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  [DONE] Variant     : {CONFIG['model_variant']}")
    print(f"  [DONE] Test Acc    : {final_acc:.2f}%  "
          f"({'✓ ≥98%' if final_acc >= 98.0 else '✗ <98%'})")
    print(f"  [DONE] Weights     : {CONFIG['model_save_path']}")
    print(f"  [DONE] Metadata    : {CONFIG['metadata_save_path']}")
    print(f"  [DONE] Run ID      : {metadata.run_id}")
    print(f"{'='*65}")


if __name__ == "__main__":
    main()

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.81MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 155kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.63MB/s]


[DATA] Train: 60000  |  Test: 10000
[DATA] Batch size: 64  |  Input: (1, 28, 28)  |  Classes: 10

[MODEL] CNN_BNS_HW — Dual ConvBlock (no BatchNorm, bias=True)
  type            : CNN_BNS_HW (dual-conv)
  batch_norm      : False
  conv_bias       : True
  conv_block1     : 1→32 ch, k=3, pad=1
  conv_block2     : 32→64 ch, k=3, pad=1
  flatten_dim     : 3136
  fc1             : 3136→256
  fc2_out         : 256→10
  total_params    : 824458

  Training — 10 epochs | variant: dual
  Epoch [1/10] Batch [93/938] Loss: 0.2266 | Running Acc: 80.16%
  Epoch [1/10] Batch [186/938] Loss: 0.1785 | Running Acc: 86.95%
  Epoch [1/10] Batch [279/938] Loss: 0.1881 | Running Acc: 89.83%
  Epoch [1/10] Batch [372/938] Loss: 0.1709 | Running Acc: 91.36%
  Epoch [1/10] Batch [465/938] Loss: 0.0448 | Running Acc: 92.35%
  Epoch [1/10] Batch [558/938] Loss: 0.0974 | Running Acc: 93.09%
  Epoch [1/10] Batch [651/938] Loss: 0.1035 | Running Acc: 93.68%
  Epoch [1/10] Batch [744/938] Loss: 0.1256 | Running Ac